In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

In [2]:
X_train = pd.read_csv("../data/processed/X_train.csv")
X_test = pd.read_csv("../data/processed/X_test.csv")

y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()

preprocessor = joblib.load("../models/preprocessor.pkl")

In [3]:
X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [4]:
def evaluate_model(model, X_train, X_test, y_train, y_test):
    
    # Train model
    model.fit(X_train, y_train)
    
    # Predictions
    y_pred = model.predict(X_test)
    
    # Probabilities (if supported)
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:, 1]
        roc_auc = roc_auc_score(y_test, y_prob)
    else:
        roc_auc = None
    
    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    return {
        "Model": model.__class__.__name__,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "ROC AUC": roc_auc
    }

In [5]:
lr = LogisticRegression(max_iter=1000)

lr_result = evaluate_model(
    lr,
    X_train_processed,
    X_test_processed,
    y_train,
    y_test
)

print(lr_result)

{'Model': 'LogisticRegression', 'Accuracy': 0.9616749467707594, 'Precision': 0.9545454545454546, 'Recall': 0.8983957219251337, 'F1 Score': 0.9256198347107438, 'ROC AUC': 0.9917951897491539}


In [6]:
dt = DecisionTreeClassifier(random_state=42)

dt_result = evaluate_model(
    dt,
    X_train_processed,
    X_test_processed,
    y_train,
    y_test
)

print(dt_result)

{'Model': 'DecisionTreeClassifier', 'Accuracy': 0.9488999290276792, 'Precision': 0.9148351648351648, 'Recall': 0.8903743315508021, 'F1 Score': 0.9024390243902439, 'ROC AUC': 0.930211320364773}


In [7]:
rf = RandomForestClassifier(
    random_state=42
)

rf_result = evaluate_model(
    rf,
    X_train_processed,
    X_test_processed,
    y_train,
    y_test
)

print(rf_result)

{'Model': 'RandomForestClassifier', 'Accuracy': 0.9318665720369056, 'Precision': 0.9455128205128205, 'Recall': 0.7887700534759359, 'F1 Score': 0.8600583090379009, 'ROC AUC': 0.9650933891343099}


In [8]:
results = pd.DataFrame([
    lr_result,
    dt_result,
    rf_result
])

results

,Model,Accuracy,Precision,Recall,F1 Score,ROC AUC
0,LogisticRegression,0.961675,0.954545,0.898396,0.925620,0.991795
1,DecisionTreeClassifier,0.948900,0.914835,0.890374,0.902439,0.930211
2,RandomForestClassifier,0.931867,0.945513,0.788770,0.860058,0.965093


In [9]:
results.sort_values(
    by="F1 Score",
    ascending=False
)

,Model,Accuracy,Precision,Recall,F1 Score,ROC AUC
0,LogisticRegression,0.961675,0.954545,0.898396,0.925620,0.991795
1,DecisionTreeClassifier,0.948900,0.914835,0.890374,0.902439,0.930211
2,RandomForestClassifier,0.931867,0.945513,0.788770,0.860058,0.965093


In [10]:
joblib.dump(
    lr,
    "../models/churn_prediction_model.pkl"
)

['../models/churn_prediction_model.pkl']